In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.6 MB/s eta 0:00:00a 0:00:01


# 02 — Training Replicas

For each window pair (A, B) this notebook:

1. Tunes hyperparameters **per window** using Optuna with TimeSeriesSplit, optimising PR-AUC.
2. Trains **R replicas** per window using stratified bootstrap sampling + different random seeds.
3. Evaluates on the common evaluation slice E_{A,B} (replica-averaged predictions, PR-AUC and ROC-AUC).
4. Identifies the **flagged set** F_{A,B} as the top K_FRAC fraction of eval instances ranked by max(p_hat_A, p_hat_B).

**Model types supported:** `'xgboost'` | `'logreg'`  ← set via `MODEL_TYPE` in the config cell.  
MLP-PLR is trained in `02b_training_replicas_mlp_plr.ipynb`.

## Per-replica preprocessing (Homesite specifics)

Mirrors Shoppers' per-replica `StandardScaler` pattern, extended for the `X_cat` group introduced in notebook 00:

- A fresh `OrdinalEncoder(min_frequency=1/100, handle_unknown='use_encoded_value', unknown_value=-1)` is fit on **each replica's bootstrap sample** of categorical columns. Categories appearing in <1% of the bootstrap collapse to a single bucket; categories absent from the bootstrap that show up at inference time get `-1`.
- The encoded categoricals are `np.hstack`-ed onto the (numeric + binary) `X` matrix, producing a combined float32 model input.
- The fitted encoder is bundled alongside the model artefact (`encoder_r{r}.joblib`) so it can be reapplied during explanation extraction (notebooks 03 / 03b) and drift analysis (notebook 04).
- A `reference_encoder.joblib` (fit on the full window A's categoricals) is also saved per pair, parallel to `reference_scaler.joblib`, for cross-pair drift comparisons in notebook 04.

## ES validation correctness

For XGBoost, the early-stopping validation slice is the chronological **tail** of the training window, held out **before** bootstrapping (Shoppers convention). This guarantees ES monitors out-of-time data rather than a random slice of a shuffled bootstrap.

**Input:** `data/processed/`, `data/windows/window_config.json`  
**Output per pair:** `data/models/{model_type}/pair_{pid:02d}/`
- `replicas_A/model_r{r}.joblib`, `replicas_B/model_r{r}.joblib` — R fitted models
- `replicas_A/encoder_r{r}.joblib`, `replicas_B/encoder_r{r}.joblib` — R fitted OrdinalEncoders
- `replicas_A/seeds_r{r}.json`, `replicas_B/seeds_r{r}.json` — exact bootstrap/model seeds
- `hparams_A.json`, `hparams_B.json` — tuned hyperparameters
- `reference_scaler.joblib` — StandardScaler fit on window A's numeric features (used by notebook 04 for covariate drift, and by LR coefficient reprojection)
- `reference_encoder.joblib` — OrdinalEncoder fit on window A's categorical features (used by notebook 04 for cross-pair drift comparison)
- `predictions.npz` — see schema below
- `coef_A.npy`, `coef_B.npy` *(LR only)* — replica coefficient tensors (sliced to num+bin only; cat coefficients differ across replicas due to per-replica encoders), shape `(R, n_num + n_bin)`
- `coef_A_ref.npy`, `coef_B_ref.npy` *(LR only)* — reprojected to the common pair-level reference basis; shape `(R, n_num + n_bin)`
- `coef_A_full_ref.npy`, `coef_B_full_ref.npy` *(LR only)* — single deterministic full-window LR fit, reprojected; shape `(n_num + n_bin,)`
- `full_model_A.joblib`, `full_model_B.joblib` *(LR only)* — fitted full-window LR pipelines (StandardScaler + LogisticRegression)
- `full_model_A_encoder.joblib`, `full_model_B_encoder.joblib` *(LR only)* — encoders used to build the full-window LR fits

For XGBoost runs only:
- `replicas_A/training_log_r{r}.csv`, `replicas_B/training_log_r{r}.csv` — per-replica eval-metric curve

**`predictions.npz` schema (unchanged across model types):**
| Key | Shape | Meaning |
|---|---|---|
| `preds_A` | `(R, n_eval)` | per-replica positive-class probability, window A |
| `preds_B` | `(R, n_eval)` | per-replica positive-class probability, window B |
| `p_hat_A` | `(n_eval,)` | replica-averaged probability, window A |
| `p_hat_B` | `(n_eval,)` | replica-averaged probability, window B |
| `flagged_idx` | `(n_flagged,)` | local positions within `idx_eval` of the flagged set |
| `Y_eval` | `(n_eval,)` | true labels |
| `pr_auc_A` | scalar | average precision of `p_hat_A` |
| `pr_auc_B` | scalar | average precision of `p_hat_B` |
| `roc_auc_A` | scalar | ROC-AUC of `p_hat_A` |
| `roc_auc_B` | scalar | ROC-AUC of `p_hat_B` |
| `per_replica_pr_auc_A` | `(R,)` | PR-AUC of each individual replica, window A |
| `per_replica_pr_auc_B` | `(R,)` | PR-AUC of each individual replica, window B |

---

**Replica design:** Each replica differs by (1) a different random seed and (2) a stratified bootstrap sample drawn from the chronologically-trimmed training window.

In [3]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

WORKSPACE = Path('/content/drive/MyDrive/Thesis/Homesite_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Fixed parameters (not tuned) ──────────────────────────────────────
XGB_FIXED = dict(
    tree_method = 'hist',
    eval_metric = 'aucpr',
    verbosity   = 0,
    n_jobs      = -1,
    objective   = 'binary:logistic',
)

import sklearn as _sklearn
_sk_ver = tuple(int(x) for x in _sklearn.__version__.split('.')[:2])
_lr_solver = 'newton-cholesky' if _sk_ver >= (1, 2) else 'lbfgs'

LR_FIXED = dict(
    solver   = _lr_solver,
    max_iter = 500,
)

# ── Tuning configuration ───────────────────────────────────────────────
# Compromise setting: 30 Optuna trials with Shoppers' deeper ES rounds and boost cap.
N_TRIALS       = 30    # Optuna trials per window (XGBoost)
N_TRIALS_LR    = 30    # Optuna trials per window (LogisticRegression)
CV_N_SPLITS    = 3     # TimeSeriesSplit folds inside each window
ES_ROUNDS      = 50    # early-stopping rounds (XGBoost only)
MAX_BOOST_RND  = 2000  # hard cap on n_estimators (XGBoost only)
VAL_TAIL_FRAC  = 0.15  # chronological window tail held out for XGBoost early stopping

# ── Categorical encoding configuration ─────────────────────────────────
# Mirrors TabReD's homesite.py: categories with <1% frequency collapse to a
# single bucket. handle_unknown='use_encoded_value' keeps inference safe when
# the eval slice contains categories the bootstrap did not sample.
CAT_MIN_FREQ  = 0.01
CAT_UNK_VALUE = -1

print('Imports OK')
print(f'LR solver: {_lr_solver}  (sklearn {_sklearn.__version__})')

Imports OK
LR solver: newton-cholesky  (sklearn 1.6.1)


In [4]:
# ── Model-type configuration — change this between runs ───────────────
MODEL_TYPE = 'xgboost'   # 'xgboost' or 'logreg'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'MODEL_TYPE : {MODEL_TYPE}')
print(f'MODEL_DIR  : {MODEL_DIR}')

assert MODEL_TYPE in {'xgboost', 'logreg'}, (
    f"MODEL_TYPE must be either 'xgboost' or 'logreg'. Got: {MODEL_TYPE}"
)

MODEL_TYPE : xgboost
MODEL_DIR  : /content/drive/MyDrive/Thesis/Homesite_workspace/data/models/xgboost


In [5]:
# ── Load processed data ────────────────────────────────────────────────
X_df = pd.read_parquet(PROC_DIR / 'X.parquet')
X    = X_df.values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

# Categoricals are stored as raw strings (object dtype).
X_cat_df = pd.read_parquet(PROC_DIR / 'X_cat.parquet')
X_cat    = np.asarray(X_cat_df, dtype=object)

with open(PROC_DIR / 'feature_names.json') as f:
    feat_names = json.load(f)
num_cols = feat_names['num']                             # numeric feature names
bin_cols = feat_names['bin']                             # binary feature names
cat_cols = feat_names['cat']                             # categorical feature names
num_col_idx = [X_df.columns.get_loc(c) for c in num_cols]  # positions in X (for reference scaler)

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R       = config['parameters']['R']
K_FRAC  = config['parameters']['K_FRAC']
pairs   = config['pairs']

n_num_bin = X.shape[1]                       # columns in X.parquet (= num + bin)
n_cat     = X_cat.shape[1]                   # columns in X_cat.parquet
n_total   = n_num_bin + n_cat                # combined feature count fed to models

print(f'X     : {X.shape}     ({len(num_cols)} num + {len(bin_cols)} bin)')
print(f'X_cat : {X_cat.shape}  ({len(cat_cols)} cat features as raw strings)')
print(f'Y     : {Y.shape}')
print(f'R={R}, K_FRAC={K_FRAC}, {len(pairs)} window pairs')
print(f'Combined feature count fed to models: {n_total}')

assert X.shape[0] == Y.shape[0] == X_cat.shape[0], 'X / Y / X_cat row counts inconsistent.'
assert X.shape[1] == len(feat_names['all']),       'X.parquet does not match feature_names["all"].'
assert list(X_df.columns)     == feat_names['all'], 'X.parquet column order mismatch with feature_names["all"].'
assert list(X_cat_df.columns) == cat_cols,          'X_cat.parquet column order mismatch with feature_names["cat"].'
assert len(pairs) > 0, 'No rolling-window pairs found.'

X     : (260753, 428)     (253 num + 175 bin)
X_cat : (260753, 23)  (23 cat features as raw strings)
Y     : (260753,)
R=2, K_FRAC=0.1, 4 window pairs
Combined feature count fed to models: 451


## Helper functions

In [6]:
def stratified_bootstrap(idx: np.ndarray, Y: np.ndarray, seed: int) -> np.ndarray:
    """Stratified bootstrap: sample with replacement, preserving class ratio.

    Returns an array of row indices (into the global X/Y) with length equal to len(idx),
    containing the same number of positives and negatives as the input slice.
    """
    rng = np.random.default_rng(seed)
    pos = idx[Y[idx] == 1]
    neg = idx[Y[idx] == 0]
    boot_pos = rng.choice(pos, size=len(pos), replace=True)
    boot_neg = rng.choice(neg, size=len(neg), replace=True)
    out = np.concatenate([boot_pos, boot_neg])
    rng.shuffle(out)  # mix positives and negatives so any tail split is not all-positive
    return out


def make_encoder() -> OrdinalEncoder:
    """Fresh OrdinalEncoder configured with the TabReD-style frequency threshold.

    Categories with <1% relative frequency at fit time get collapsed to a single
    bucket; unseen categories at transform time get unknown_value (-1).
    """
    return OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=CAT_UNK_VALUE,
        min_frequency=CAT_MIN_FREQ,
        encoded_missing_value=CAT_UNK_VALUE,
    )


def encode_and_stack(X_num_part: np.ndarray,
                     X_cat_str_part: np.ndarray,
                     encoder: OrdinalEncoder,
                     fit: bool) -> np.ndarray:
    """Encode the categorical part and horizontally stack with the numeric+binary part.

    The combined matrix has columns in the order [num + bin | cat_encoded] so that
    `num_col_idx` (positions in X) still indexes the numeric features in the combined
    matrix without modification.
    """
    if fit:
        X_cat_enc = encoder.fit_transform(X_cat_str_part)
    else:
        X_cat_enc = encoder.transform(X_cat_str_part)
    return np.hstack([X_num_part, X_cat_enc.astype(np.float32)])


def _build_xgb(params: dict, seed: int, early_stopping_rounds: int = None) -> XGBClassifier:
    """Construct an XGBClassifier with fixed settings + tuned params + a given seed."""
    full = {**XGB_FIXED, **params, 'random_state': seed, 'n_estimators': MAX_BOOST_RND}
    if early_stopping_rounds is not None:
        full['early_stopping_rounds'] = early_stopping_rounds
    return XGBClassifier(**full)


def tune_hyperparameters_xgboost(window_idx: np.ndarray,
                                  X_all: np.ndarray,
                                  X_cat_all: np.ndarray,
                                  Y_all: np.ndarray,
                                  study_seed: int,
                                  n_trials: int = N_TRIALS) -> dict:
    """Optuna tuning of XGBoost hyperparameters on the given training window.

    Uses TimeSeriesSplit(CV_N_SPLITS) over the chronological order of window_idx.
    A fresh OrdinalEncoder is fit per fold on the fold's training portion only,
    so categorical encoding is leak-free across folds. Objective: mean PR-AUC.
    """
    X_win     = X_all[window_idx]      # preserves chronological step-sorted order
    X_cat_win = X_cat_all[window_idx]
    Y_win     = Y_all[window_idx]

    tscv = TimeSeriesSplit(n_splits=CV_N_SPLITS)

    def objective(trial: optuna.Trial) -> float:
        params = {
            'max_depth':        trial.suggest_int('max_depth', 4, 10),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'gamma':            trial.suggest_float('gamma', 1e-3, 10.0, log=True),
        }
        pr_aucs = []
        for fold, (tr, va) in enumerate(tscv.split(X_win)):
            enc  = make_encoder()
            X_tr = encode_and_stack(X_win[tr], X_cat_win[tr], enc, fit=True)
            X_va = encode_and_stack(X_win[va], X_cat_win[va], enc, fit=False)
            model = _build_xgb(params, seed=study_seed + fold,
                               early_stopping_rounds=ES_ROUNDS)
            model.fit(X_tr, Y_win[tr], eval_set=[(X_va, Y_win[va])], verbose=False)
            p = model.predict_proba(X_va)[:, 1]
            pr_aucs.append(average_precision_score(Y_win[va], p))
        return float(np.mean(pr_aucs))

    sampler = TPESampler(seed=study_seed)
    study   = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return dict(study.best_trial.params)


def tune_hyperparameters_logreg(window_idx: np.ndarray,
                                 X_all: np.ndarray,
                                 X_cat_all: np.ndarray,
                                 Y_all: np.ndarray,
                                 study_seed: int,
                                 n_trials: int = N_TRIALS_LR) -> dict:
    """Optuna tuning of LogisticRegression on the given training window.

    Tunes C only; penalty fixed to 'l2'. Per-fold OrdinalEncoder fit on the fold's
    training portion. Pipeline = StandardScaler → LogisticRegression so the per-fold
    scaler also stays leakage-free.
    """
    X_win     = X_all[window_idx]
    X_cat_win = X_cat_all[window_idx]
    Y_win     = Y_all[window_idx]

    tscv = TimeSeriesSplit(n_splits=CV_N_SPLITS)

    def objective(trial: optuna.Trial) -> float:
        params = {
            'C':       trial.suggest_float('C', 1e-3, 10.0, log=True),
            'penalty': 'l2',
        }
        pr_aucs = []
        for fold, (tr, va) in enumerate(tscv.split(X_win)):
            enc  = make_encoder()
            X_tr = encode_and_stack(X_win[tr], X_cat_win[tr], enc, fit=True)
            X_va = encode_and_stack(X_win[va], X_cat_win[va], enc, fit=False)
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('model',  LogisticRegression(**{**LR_FIXED, **params,
                                                 'random_state': study_seed + fold})),
            ])
            pipe.fit(X_tr, Y_win[tr])
            p = pipe.predict_proba(X_va)[:, 1]
            pr_aucs.append(average_precision_score(Y_win[va], p))
        return float(np.mean(pr_aucs))

    sampler = TPESampler(seed=study_seed)
    study   = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return dict(study.best_trial.params)


def train_replica_with_es(X_train_combined: np.ndarray,
                           Y_train: np.ndarray,
                           X_val_combined: np.ndarray,
                           Y_val: np.ndarray,
                           params: dict,
                           model_seed: int):
    """Train one XGBoost replica on a bootstrap sample with early stopping.

    Inputs are the already-combined (numeric + binary + encoded categorical) matrices.
    Returns (model, training_log_df).
    """
    model = _build_xgb(params, seed=model_seed, early_stopping_rounds=ES_ROUNDS)
    model.fit(X_train_combined, Y_train,
              eval_set=[(X_val_combined, Y_val)], verbose=False)
    evals_result = model.evals_result()
    val_curve = evals_result.get('validation_0', {}).get('aucpr', [])
    training_log = pd.DataFrame({
        'iteration': np.arange(1, len(val_curve) + 1),
        'val_aucpr': val_curve,
    })
    return model, training_log


def train_replica_logreg(X_train_combined: np.ndarray,
                          Y_train: np.ndarray,
                          params: dict,
                          model_seed: int) -> Pipeline:
    """Train one LR replica. Returns a fitted Pipeline(StandardScaler, LogisticRegression)."""
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  LogisticRegression(**{**LR_FIXED, **params, 'random_state': model_seed})),
    ])
    pipe.fit(X_train_combined, Y_train)
    return pipe


def lr_coef_to_reference_basis(pipe: Pipeline,
                                reference_scaler: StandardScaler,
                                num_col_idx: list,
                                n_num_bin: int) -> np.ndarray:
    """Convert a fitted LR pipeline's coefficients into a common pair-level reference basis.

    The pipeline was trained on a (n_num + n_bin + n_cat) combined matrix.  Cat coefficients
    are stripped because each replica's encoder maps categories to different integers, so
    the cat-feature coefficients are not directly comparable across replicas.  Numeric
    coefficients are reprojected per the reference scaler's σ; binary coefficients stay raw.
    Returns a `(n_num + n_bin,)` float32 array.
    """
    scaler = pipe.named_steps['scaler']
    model  = pipe.named_steps['model']

    beta_std_full = model.coef_.ravel().astype(np.float64)
    beta_std      = beta_std_full[:n_num_bin]                  # drop cat coefficients
    train_scale   = scaler.scale_[:n_num_bin].astype(np.float64)
    train_scale_safe = np.where(train_scale == 0.0, 1.0, train_scale)

    beta_raw = beta_std / train_scale_safe
    beta_ref = beta_raw.copy()
    ref_scale = reference_scaler.scale_.astype(np.float64)

    for pos_in_ref, j in enumerate(num_col_idx):
        beta_ref[j] = beta_raw[j] * ref_scale[pos_in_ref]

    return beta_ref.astype(np.float32)


def train_full_window_logreg(X_train_combined: np.ndarray,
                              Y_train: np.ndarray,
                              params: dict,
                              model_seed: int) -> Pipeline:
    """Deterministic full-window LR fit (no bootstrap)."""
    return train_replica_logreg(X_train_combined, Y_train, params, model_seed)


def predict_proba_pos(model, X_in: np.ndarray) -> np.ndarray:
    """Return probability of positive class."""
    return model.predict_proba(X_in)[:, 1]


def compute_flagged_topk(p_hat_A: np.ndarray,
                          p_hat_B: np.ndarray,
                          k_frac: float) -> np.ndarray:
    """Top-K% flagged set: the top k_frac fraction ranked by max(p_hat_A, p_hat_B)."""
    assert p_hat_A.shape == p_hat_B.shape
    n = len(p_hat_A)
    k = max(int(np.ceil(k_frac * n)), 1)
    score = np.maximum(p_hat_A, p_hat_B)
    topk_unsorted = np.argpartition(-score, k - 1)[:k]
    return np.sort(topk_unsorted)


def assert_binary_slice(y, name):
    assert len(np.unique(y)) == 2, f'{name} contains only one class.'


print('Helpers defined.')

Helpers defined.


## Main training loop — one iteration per window pair

In [7]:
SEED_BASE = 42

# Full run-configuration fingerprint used for staleness detection.
# If any of these settings change, existing predictions are treated as stale and the pair is re-run.
CURRENT_RUN_PARAMS = {
    'model_type':  MODEL_TYPE,
    'seed_base':   SEED_BASE,

    'window_parameters': {
        'L':           int(config['parameters']['L']),
        'S':           int(config['parameters']['S']),
        'H':           int(config['parameters']['H']),
        'R':           int(R),
        'K_FRAC':      float(K_FRAC),
        'PAIR_STRIDE': int(config['parameters'].get('PAIR_STRIDE', 1)),
    },

    'cv_n_splits':   int(CV_N_SPLITS),
    'cat_min_freq':  float(CAT_MIN_FREQ),
    'cat_unk_value': int(CAT_UNK_VALUE),
}

if MODEL_TYPE == 'logreg':
    CURRENT_RUN_PARAMS['lr_reference_basis_schema']    = 'coef_ref_basis_v1_homesite'
    CURRENT_RUN_PARAMS['lr_full_window_reference_fit'] = True
    CURRENT_RUN_PARAMS['n_trials_logreg']              = int(N_TRIALS_LR)
    CURRENT_RUN_PARAMS['lr_fixed']                     = LR_FIXED
else:  # xgboost
    CURRENT_RUN_PARAMS.update({
        'n_trials_xgboost': int(N_TRIALS),
        'es_rounds':        int(ES_ROUNDS),
        'max_boost_round':  int(MAX_BOOST_RND),
        'val_tail_frac':    float(VAL_TAIL_FRAC),
        'xgb_fixed':        XGB_FIXED,
    })

performance_log = []

for p in pairs:
    pid             = p['pair_id']
    pair_dir        = MODEL_DIR / f'pair_{pid:02d}'
    pred_file       = pair_dir / 'predictions.npz'
    run_params_file = pair_dir / 'run_params.json'

    # ── Skip-if-done: validate full run configuration against saved run_params ──
    if pred_file.exists() and run_params_file.exists():
        with open(run_params_file) as f:
            saved = json.load(f)
        params_match = (saved == CURRENT_RUN_PARAMS)

        required_lr_files = [
            pair_dir / 'coef_A_ref.npy',
            pair_dir / 'coef_B_ref.npy',
            pair_dir / 'coef_A_full_ref.npy',
            pair_dir / 'coef_B_full_ref.npy',
            pair_dir / 'full_model_A.joblib',
            pair_dir / 'full_model_B.joblib',
        ]
        if MODEL_TYPE == 'logreg' and not all(pp.exists() for pp in required_lr_files):
            print(f'Pair {pid:02d}: LR coefficient artefacts missing — re-running.')
        else:
            if params_match:
                data = np.load(pred_file)
                if all(k in data.files for k in ('roc_auc_A', 'roc_auc_B')):
                    print(f'Pair {pid:02d}: already done, skipping.')
                    performance_log.append({
                        'pair_id':   pid,
                        'pr_auc_A':  float(data['pr_auc_A']),
                        'pr_auc_B':  float(data['pr_auc_B']),
                        'roc_auc_A': float(data['roc_auc_A']),
                        'roc_auc_B': float(data['roc_auc_B']),
                        'n_flagged': int(data['flagged_idx'].shape[0]),
                    })
                    continue
                else:
                    print(f'Pair {pid:02d}: stale predictions.npz (missing roc_auc keys) — re-running.')
            else:
                changed_keys = [k for k in CURRENT_RUN_PARAMS if saved.get(k) != CURRENT_RUN_PARAMS[k]]
                print(f'Pair {pid:02d}: run_params changed in {changed_keys} — re-running.')
    elif pred_file.exists():
        print(f'Pair {pid:02d}: run_params.json missing — re-running.')

    print(f'\n── [{MODEL_TYPE}] Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]}  '
          f'eval={p["eval_start_label"]}→{p["eval_end_label"]}  '
          f'|A|={p["n_train_A"]:,} |B|={p["n_train_B"]:,} |eval|={p["n_eval"]:,} ──')

    idx_A    = np.array(p['idx_A'],    dtype=np.int64)
    idx_B    = np.array(p['idx_B'],    dtype=np.int64)
    idx_eval = np.array(p['idx_eval'], dtype=np.int64)

    assert len(set(idx_A.tolist()) & set(idx_eval.tolist())) == 0, 'idx_A and idx_eval overlap!'
    assert len(set(idx_B.tolist()) & set(idx_eval.tolist())) == 0, 'idx_B and idx_eval overlap!'

    X_eval     = X[idx_eval]
    X_cat_eval = X_cat[idx_eval]
    Y_eval     = Y[idx_eval]

    assert_binary_slice(Y[idx_A], f'pair {pid:02d} window A')
    assert_binary_slice(Y[idx_B], f'pair {pid:02d} window B')
    assert_binary_slice(Y_eval,   f'pair {pid:02d} eval slice')

    pair_dir.mkdir(parents=True, exist_ok=True)
    dir_A = pair_dir / 'replicas_A'
    dir_B = pair_dir / 'replicas_B'
    dir_A.mkdir(exist_ok=True)
    dir_B.mkdir(exist_ok=True)

    # ── Reference scaler: window-A numeric features (used by nb 04 + LR coefficient reprojection) ──
    ref_scaler = StandardScaler()
    ref_scaler.fit(X[idx_A][:, num_col_idx])
    joblib.dump(ref_scaler, pair_dir / 'reference_scaler.joblib')

    # ── Reference encoder: window-A categorical features (used by nb 04 for cat drift) ──
    ref_encoder = make_encoder()
    ref_encoder.fit(X_cat[idx_A])
    joblib.dump(ref_encoder, pair_dir / 'reference_encoder.joblib')

    # ── Tune hyperparameters per window ──
    print('  Tuning window A ...')
    if MODEL_TYPE == 'xgboost':
        hparams_A = tune_hyperparameters_xgboost(idx_A, X, X_cat, Y,
                                                  study_seed=SEED_BASE + pid * 10 + 1)
    else:
        hparams_A = tune_hyperparameters_logreg(idx_A, X, X_cat, Y,
                                                 study_seed=SEED_BASE + pid * 10 + 1)
    with open(pair_dir / 'hparams_A.json', 'w') as f:
        json.dump(hparams_A, f, indent=2)
    print(f'    best A params: {hparams_A}')

    print('  Tuning window B ...')
    if MODEL_TYPE == 'xgboost':
        hparams_B = tune_hyperparameters_xgboost(idx_B, X, X_cat, Y,
                                                  study_seed=SEED_BASE + pid * 10 + 2)
    else:
        hparams_B = tune_hyperparameters_logreg(idx_B, X, X_cat, Y,
                                                 study_seed=SEED_BASE + pid * 10 + 2)
    with open(pair_dir / 'hparams_B.json', 'w') as f:
        json.dump(hparams_B, f, indent=2)
    print(f'    best B params: {hparams_B}')

    # ── Chronological ES validation split (XGBoost only; done before bootstrapping) ──
    if MODEL_TYPE == 'xgboost':
        n_es_A     = max(int(np.ceil(len(idx_A) * VAL_TAIL_FRAC)), 1)
        idx_A_boot = idx_A[:-n_es_A]
        idx_A_es   = idx_A[-n_es_A:]
        n_es_B     = max(int(np.ceil(len(idx_B) * VAL_TAIL_FRAC)), 1)
        idx_B_boot = idx_B[:-n_es_B]
        idx_B_es   = idx_B[-n_es_B:]

        assert_binary_slice(Y[idx_A_es],   f'pair {pid:02d} A ES tail')
        assert_binary_slice(Y[idx_B_es],   f'pair {pid:02d} B ES tail')
        assert_binary_slice(Y[idx_A_boot], f'pair {pid:02d} A bootstrap source')
        assert_binary_slice(Y[idx_B_boot], f'pair {pid:02d} B bootstrap source')
    else:
        idx_A_boot = idx_A   # logreg has no early stopping
        idx_B_boot = idx_B

    # ── Train R replicas for window A ──
    preds_A          = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_A = np.zeros(R, dtype=np.float32)
    if MODEL_TYPE == 'logreg':
        coef_A     = np.zeros((R, n_num_bin), dtype=np.float32)
        coef_A_ref = np.zeros((R, n_num_bin), dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + r * 2          # A-bootstrap seeds are even
        model_seed = SEED_BASE + pid * 10_000 + r * 2 + 1      # A-model seeds are odd
        boot_idx   = stratified_bootstrap(idx_A_boot, Y, seed=boot_seed)

        # Fresh per-replica encoder fit on the bootstrap categorical slice.
        encoder = make_encoder()
        X_tr_combined = encode_and_stack(X[boot_idx], X_cat[boot_idx], encoder, fit=True)
        Y_tr = Y[boot_idx]

        if MODEL_TYPE == 'xgboost':
            X_va_combined   = encode_and_stack(X[idx_A_es], X_cat[idx_A_es], encoder, fit=False)
            X_eval_combined = encode_and_stack(X_eval,      X_cat_eval,      encoder, fit=False)
            model, training_log = train_replica_with_es(
                X_tr_combined, Y_tr, X_va_combined, Y[idx_A_es], hparams_A, model_seed,
            )
            preds_A[r] = predict_proba_pos(model, X_eval_combined)
            joblib.dump(model, dir_A / f'model_r{r}.joblib', compress=3)
            training_log.to_csv(dir_A / f'training_log_r{r}.csv', index=False)
        else:  # logreg
            X_eval_combined = encode_and_stack(X_eval, X_cat_eval, encoder, fit=False)
            pipeline = train_replica_logreg(X_tr_combined, Y_tr, hparams_A, model_seed)
            preds_A[r] = pipeline.predict_proba(X_eval_combined)[:, 1]
            beta_full   = pipeline.named_steps['model'].coef_.ravel().astype(np.float32)
            coef_A[r]   = beta_full[:n_num_bin]
            coef_A_ref[r] = lr_coef_to_reference_basis(
                pipe=pipeline, reference_scaler=ref_scaler,
                num_col_idx=num_col_idx, n_num_bin=n_num_bin,
            )
            joblib.dump(pipeline, dir_A / f'model_r{r}.joblib', compress=3)

        # Bundle the per-replica encoder alongside the model so explanations and drift
        # analyses downstream can reproduce the exact training-time encoding.
        joblib.dump(encoder, dir_A / f'encoder_r{r}.joblib', compress=3)

        per_rep_pr_auc_A[r] = average_precision_score(Y_eval, preds_A[r])
        with open(dir_A / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)
        print(f'  A replica {r}: PR-AUC = {per_rep_pr_auc_A[r]:.4f}')

    # ── Train R replicas for window B ──
    preds_B          = np.zeros((R, len(idx_eval)), dtype=np.float32)
    per_rep_pr_auc_B = np.zeros(R, dtype=np.float32)
    if MODEL_TYPE == 'logreg':
        coef_B     = np.zeros((R, n_num_bin), dtype=np.float32)
        coef_B_ref = np.zeros((R, n_num_bin), dtype=np.float32)

    for r in range(R):
        boot_seed  = SEED_BASE + pid * 10_000 + 5_000 + r * 2
        model_seed = SEED_BASE + pid * 10_000 + 5_000 + r * 2 + 1
        boot_idx   = stratified_bootstrap(idx_B_boot, Y, seed=boot_seed)

        encoder = make_encoder()
        X_tr_combined = encode_and_stack(X[boot_idx], X_cat[boot_idx], encoder, fit=True)
        Y_tr = Y[boot_idx]

        if MODEL_TYPE == 'xgboost':
            X_va_combined   = encode_and_stack(X[idx_B_es], X_cat[idx_B_es], encoder, fit=False)
            X_eval_combined = encode_and_stack(X_eval,      X_cat_eval,      encoder, fit=False)
            model, training_log = train_replica_with_es(
                X_tr_combined, Y_tr, X_va_combined, Y[idx_B_es], hparams_B, model_seed,
            )
            preds_B[r] = predict_proba_pos(model, X_eval_combined)
            joblib.dump(model, dir_B / f'model_r{r}.joblib', compress=3)
            training_log.to_csv(dir_B / f'training_log_r{r}.csv', index=False)
        else:
            X_eval_combined = encode_and_stack(X_eval, X_cat_eval, encoder, fit=False)
            pipeline = train_replica_logreg(X_tr_combined, Y_tr, hparams_B, model_seed)
            preds_B[r] = pipeline.predict_proba(X_eval_combined)[:, 1]
            beta_full   = pipeline.named_steps['model'].coef_.ravel().astype(np.float32)
            coef_B[r]   = beta_full[:n_num_bin]
            coef_B_ref[r] = lr_coef_to_reference_basis(
                pipe=pipeline, reference_scaler=ref_scaler,
                num_col_idx=num_col_idx, n_num_bin=n_num_bin,
            )
            joblib.dump(pipeline, dir_B / f'model_r{r}.joblib', compress=3)

        joblib.dump(encoder, dir_B / f'encoder_r{r}.joblib', compress=3)

        per_rep_pr_auc_B[r] = average_precision_score(Y_eval, preds_B[r])
        with open(dir_B / f'seeds_r{r}.json', 'w') as f:
            json.dump({'bootstrap_seed': int(boot_seed),
                       'model_seed':     int(model_seed)}, f, indent=2)
        print(f'  B replica {r}: PR-AUC = {per_rep_pr_auc_B[r]:.4f}')

    # ── LR coefficient saving + full-window reference fits ──
    if MODEL_TYPE == 'logreg':
        np.save(pair_dir / 'coef_A.npy',     coef_A)
        np.save(pair_dir / 'coef_A_ref.npy', coef_A_ref)
        np.save(pair_dir / 'coef_B.npy',     coef_B)
        np.save(pair_dir / 'coef_B_ref.npy', coef_B_ref)
        print(f'  Saved coef_A.npy / coef_A_ref.npy   shape={coef_A.shape}')
        print(f'  Saved coef_B.npy / coef_B_ref.npy   shape={coef_B.shape}')

        print('  Training full-window LR reference models ...')
        full_seed_A = SEED_BASE + pid * 10_000 + 999
        full_seed_B = SEED_BASE + pid * 10_000 + 5_999

        full_enc_A   = make_encoder()
        X_full_A     = encode_and_stack(X[idx_A], X_cat[idx_A], full_enc_A, fit=True)
        full_pipe_A  = train_full_window_logreg(X_full_A, Y[idx_A], hparams_A, full_seed_A)

        full_enc_B   = make_encoder()
        X_full_B     = encode_and_stack(X[idx_B], X_cat[idx_B], full_enc_B, fit=True)
        full_pipe_B  = train_full_window_logreg(X_full_B, Y[idx_B], hparams_B, full_seed_B)

        coef_A_full_ref = lr_coef_to_reference_basis(
            pipe=full_pipe_A, reference_scaler=ref_scaler,
            num_col_idx=num_col_idx, n_num_bin=n_num_bin,
        )
        coef_B_full_ref = lr_coef_to_reference_basis(
            pipe=full_pipe_B, reference_scaler=ref_scaler,
            num_col_idx=num_col_idx, n_num_bin=n_num_bin,
        )

        joblib.dump(full_pipe_A, pair_dir / 'full_model_A.joblib',         compress=3)
        joblib.dump(full_pipe_B, pair_dir / 'full_model_B.joblib',         compress=3)
        joblib.dump(full_enc_A,  pair_dir / 'full_model_A_encoder.joblib', compress=3)
        joblib.dump(full_enc_B,  pair_dir / 'full_model_B_encoder.joblib', compress=3)
        np.save(pair_dir / 'coef_A_full_ref.npy', coef_A_full_ref)
        np.save(pair_dir / 'coef_B_full_ref.npy', coef_B_full_ref)

        print(f'  Saved full_model_A/B + their encoders')
        print(f'  Saved coef_A_full_ref.npy shape={coef_A_full_ref.shape}')
        print(f'  Saved coef_B_full_ref.npy shape={coef_B_full_ref.shape}')

    # ── Replica-averaged predictions and flagged set ──
    p_hat_A = preds_A.mean(axis=0)
    p_hat_B = preds_B.mean(axis=0)
    flagged_local_idx = compute_flagged_topk(p_hat_A, p_hat_B, K_FRAC)

    pr_auc_A  = average_precision_score(Y_eval, p_hat_A)
    pr_auc_B  = average_precision_score(Y_eval, p_hat_B)
    roc_auc_A = roc_auc_score(Y_eval, p_hat_A)
    roc_auc_B = roc_auc_score(Y_eval, p_hat_B)

    print(f'  Averaged: PR-AUC A = {pr_auc_A:.4f}, PR-AUC B = {pr_auc_B:.4f}')
    print(f'            ROC-AUC A = {roc_auc_A:.4f}, ROC-AUC B = {roc_auc_B:.4f}')
    print(f'  Flagged (top {K_FRAC:.0%}): {flagged_local_idx.shape[0]:,} / {len(idx_eval):,}')

    np.savez_compressed(
        pred_file,
        preds_A              = preds_A,
        preds_B              = preds_B,
        p_hat_A              = p_hat_A,
        p_hat_B              = p_hat_B,
        flagged_idx          = flagged_local_idx,
        Y_eval               = Y_eval,
        pr_auc_A             = np.float32(pr_auc_A),
        pr_auc_B             = np.float32(pr_auc_B),
        roc_auc_A            = np.float32(roc_auc_A),
        roc_auc_B            = np.float32(roc_auc_B),
        per_replica_pr_auc_A = per_rep_pr_auc_A,
        per_replica_pr_auc_B = per_rep_pr_auc_B,
    )

    with open(run_params_file, 'w') as f:
        json.dump(CURRENT_RUN_PARAMS, f, indent=2)

    performance_log.append({
        'pair_id':   pid,
        'pr_auc_A':  pr_auc_A,
        'pr_auc_B':  pr_auc_B,
        'roc_auc_A': roc_auc_A,
        'roc_auc_B': roc_auc_B,
        'n_flagged': flagged_local_idx.shape[0],
    })

print('\n✓ All window pairs processed.')

: 

: 

In [ ]:
perf_df = pd.DataFrame(performance_log)
print(perf_df.to_string(index=False))
print(f'\nMean PR-AUC  A: {perf_df["pr_auc_A"].mean():.4f}   B: {perf_df["pr_auc_B"].mean():.4f}')
print(f'Mean ROC-AUC A: {perf_df["roc_auc_A"].mean():.4f}   B: {perf_df["roc_auc_B"].mean():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(perf_df['pair_id'], perf_df['pr_auc_A'], 'o-',  label='Model A')
axes[0].plot(perf_df['pair_id'], perf_df['pr_auc_B'], 's--', label='Model B')
axes[0].set_title('PR-AUC over window pairs')
axes[0].set_xlabel('Window pair')
axes[0].set_ylabel('PR-AUC')
axes[0].set_xticks(perf_df['pair_id'])
axes[0].legend()
axes[0].set_ylim(0, 1)

axes[1].plot(perf_df['pair_id'], perf_df['roc_auc_A'], 'o-',  label='Model A')
axes[1].plot(perf_df['pair_id'], perf_df['roc_auc_B'], 's--', label='Model B')
axes[1].set_title('ROC-AUC over window pairs')
axes[1].set_xlabel('Window pair')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_xticks(perf_df['pair_id'])
axes[1].legend()
axes[1].set_ylim(0.5, 1)

axes[2].bar(perf_df['pair_id'], perf_df['n_flagged'], color='tomato')
axes[2].set_title(f'Flagged instances per pair (top {int(K_FRAC*100)}%)')
axes[2].set_xlabel('Window pair')
axes[2].set_ylabel('|F_{A,B}|')
axes[2].set_xticks(perf_df['pair_id'])

plt.tight_layout()
plt.savefig(MODEL_DIR / 'performance_summary.png', dpi=120)
plt.show()

: 

In [ ]:
perf_df.to_csv(MODEL_DIR / 'performance_log.csv', index=False)
print('Performance log saved.')

: 